# Example 2 — Titanic Survival Classification

Build a classification pipeline with missing-value handling and categorical encoding. No GPU is required.

In [ ]:
%pip -q install pandas scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42
print("Setup complete!")

## 1. Load the dataset
This public Titanic dataset contains passenger details and a survival label.

In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)
print("Shape:", df.shape)
display(df.head())

## 2. Inspect target and missing values

In [ ]:
print("Survival counts:")
print(df["survived"].value_counts(dropna=False))
print("\\nColumns with most missing values:")
print(df.isna().sum().sort_values(ascending=False).head(15))

## 3. Prepare features and target
We use a compact feature set with numeric and categorical values. The target is `survived` (0 = did not survive, 1 = survived).

In [ ]:
df = df.dropna(subset=["survived"]).copy()
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "alone"]
X = df[features]
y = df["survived"].astype(int)

numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()
print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

## 4. Preprocess data
Numeric values use median imputation and scaling. Categories use most-frequent imputation and one-hot encoding.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

## 5. Train a classifier

In [ ]:
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
model.fit(X_train, y_train)
print("Training complete!")

## 6. Evaluate

In [ ]:
pred = model.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, pred):.3f}")
print(classification_report(y_test, pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_test, pred, display_labels=["Did not survive", "Survived"], cmap="Blues"
)
plt.title("Titanic confusion matrix")
plt.tight_layout()
plt.show()

## 7. Predict for a new passenger
This is only a model prediction and should not be interpreted as a reliable statement about an individual.

In [ ]:
new_passenger = pd.DataFrame([{
    "pclass": 3, "sex": "female", "age": 25, "sibsp": 0,
    "parch": 0, "fare": 8.05, "embarked": "S", "alone": True
}])
result = int(model.predict(new_passenger)[0])
print("Prediction:", "Survived" if result == 1 else "Did not survive")

## 8. Save the model

In [ ]:
joblib.dump(model, "titanic_survival_classifier.joblib")
print("Saved titanic_survival_classifier.joblib")

## Try it yourself
1. Train a `RandomForestClassifier` and compare it with Logistic Regression.
2. Explain why categorical encoding is needed.
3. Which metric would you examine if one class is harder to predict?
4. This dataset reflects historical social conditions; predictions are not causal explanations.